In [1]:
!pip install requests python-dotenv pandas

  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached pandas-3.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached charset_normalizer-3.4.7-cp312-cp312-macosx_10_13_universal2.whl.metadata (40 kB)
  Using cached idna-3.13-py3-none-any.whl.metadata (8.0 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached numpy-2.4.4-cp312-cp312-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached requests-2.33.1-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.7-cp312-cp312-macosx_10_13_universal2.whl (311 kB)
Using cached idna-3.13-py3-none-any.whl (68 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Using cached pandas-3.0.2-cp312-cp312-macosx_11_0_arm64.whl (9.9 MB)
Using cached certifi-2026.4.22-py3-none-any.whl (135 kB)
Using cached numpy-2.4.4-cp312-cp312-macosx_14_0_arm64.whl (5.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# 1. 인증키 불러오기
import requests
import pandas as pd
import json
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("PUBLIC_DATA_API_KEY")

if API_KEY:
    print(f"✅ API 키 로드 성공: {API_KEY[:6]}...{API_KEY[-4:]}")
else:
    print("❌ API 키를 찾을 수 없어. .env 파일 확인해봐")

✅ API 키 로드 성공: H37U62...pg==


### < 대기지수 >


In [3]:
# 2. 데이터 불러오기 - air korea 대기질 조회
# 측정소 이름을 받아서 dict 로 반환하기
def get_air_quality(station_name: str = "중구") -> dict:
    """
    에어코리아 측정소별 실시간 대기질 조회
    station_name: 측정소 이름 (예: '중구', '강남구')
    """
    # 에어 코리아 api endpoint
    url = "http://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getMsrstnAcctoRltmMesureDnsty"

    params = {
        "serviceKey": API_KEY,  # .env에서 불러온 인증키
        "returnType": "json",  # 응답 형식 (json or xml)
        "numOfRows": 1,  # 가져올 데이터 개수 (1개 = 최신 1건)
        "pageNo": 1,  # 페이지 번호
        "stationName": station_name,  # 조회할 측정소 이름
        "dataTerm": "DAILY",  # 조회 기간 (DAILY=하루 / MONTH=한달 / 3MONTH=3달)
        "ver": "1.0",  # API 버전
    }
    # 실제 호출
    response = requests.get(url, params=params)  # url + params
    data = response.json()  # 출력 형태를 json -> python dict

    return data


# 테스트 실행
air_data = get_air_quality("중구")
print("📡 에어코리아 API 응답:")
print(json.dumps(air_data, ensure_ascii=False, indent=2))

📡 에어코리아 API 응답:
{
  "response": {
    "body": {
      "totalCount": 23,
      "items": [
        {
          "so2Grade": "1",
          "coFlag": null,
          "khaiValue": "63",
          "so2Value": "0.003",
          "coValue": "0.3",
          "pm25Flag": null,
          "pm10Flag": null,
          "pm10Value": "14",
          "o3Grade": "2",
          "khaiGrade": "2",
          "pm25Value": "8",
          "no2Flag": null,
          "no2Grade": "1",
          "o3Flag": null,
          "pm25Grade": "1",
          "so2Flag": null,
          "dataTime": "2026-05-10 01:00",
          "coGrade": "1",
          "no2Value": "0.014",
          "pm10Grade": "1",
          "o3Value": "0.044"
        }
      ],
      "pageNo": 1,
      "numOfRows": 1
    },
    "header": {
      "resultMsg": "NORMAL_CODE",
      "resultCode": "00"
    }
  }
}


In [ ]:
# 3. 데이터 파싱
def parse_air_quality(data: dict) -> dict:
    """
    에어코리아 응답에서 필요한 값만 추출
    """
    try:
        # 데이터 추출
        item = data["response"]["body"]["items"][0]
        # 사용하는 데이터만
        result = {
            "측정소": item.get("stationName", "-"),
            "측정시간": item.get("dataTime", "-"),
            "PM2.5": item.get("pm25Value", "-"),  # 초미세먼지 (㎍/㎥)
            "PM10": item.get("pm10Value", "-"),  # 미세먼지
            "오존(O3)": item.get("o3Value", "-"),  # 오존
            "이산화질소(NO2)": item.get("no2Value", "-"),
            "통합대기지수(CAI)": item.get("khaiValue", "-"),
            "CAI등급": item.get("khaiGrade", "-"),  # 1:좋음 2:보통 3:나쁨 4:매우나쁨
            "PM2.5등급": item.get("pm25Grade", "-"),
            "O3등급": item.get("o3Grade", "-"),
        }
        return result

    except Exception as e:
        print(f"❌ 파싱 오류: {e}")
        return {}


# 파싱 결과 확인
parsed = parse_air_quality(air_data)
print("\n✅ 파싱 결과:")
for key, value in parsed.items():
    print(f"  {key}: {value}")


✅ 파싱 결과:
  측정소: -
  측정시간: 2026-05-10 01:00
  PM2.5: 8
  PM10: 14
  오존(O3): 0.044
  이산화질소(NO2): 0.014
  통합대기지수(CAI): 63
  CAI등급: 2
  PM2.5등급: 1
  O3등급: 2


In [ ]:
# 4. 지역 확장
# 여러 지역 한번에 조회
stations = ["중구", "강남구", "마포구", "노원구", "영등포구"]

results = []
for station in stations:
    data = get_air_quality(station)
    parsed = parse_air_quality(data)
    if parsed:
        results.append(parsed)

# 데이터프레임으로 보기
df_air = pd.DataFrame(results)
print("\n📊 서울 주요 지역 대기질 현황:")
df_air


📊 서울 주요 지역 대기질 현황:


,측정소,측정시간,PM2.5,PM10,오존(O3),이산화질소(NO2),통합대기지수(CAI),CAI등급,PM2.5등급,O3등급
0,-,2026-05-10 01:00,8,14,0.044,0.014,63,2,1,2
1,-,2026-05-10 01:00,11,19,0.042,0.015,61,2,1,2
2,-,2026-05-10 01:00,7,16,0.043,0.016,61,2,1,2
3,-,2026-05-10 01:00,8,19,0.040,0.013,59,2,1,2
4,-,2026-05-10 01:00,10,18,0.045,0.017,63,2,1,2


### < 기온 / 체감온도 >


In [12]:
# 응답 원본 확인
response = requests.get(
    "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst",
    params={
        "serviceKey": API_KEY,
        "numOfRows": 10,
        "pageNo": 1,
        "dataType": "JSON",
        "base_date": "20260509",  # ← 어제 날짜로
        "base_time": "2300",      # ← 어제 2300 발표
        "nx": 60,
        "ny": 127
    }
)

print("상태코드:", response.status_code)
print("응답 원본:", response.text[:500])

상태코드: 200
응답 원본: {"response":{"header":{"resultCode":"00","resultMsg":"NORMAL_SERVICE"},"body":{"dataType":"JSON","items":{"item":[{"baseDate":"20260509","baseTime":"2300","category":"TMP","fcstDate":"20260510","fcstTime":"0000","fcstValue":"13","nx":60,"ny":127},{"baseDate":"20260509","baseTime":"2300","category":"UUU","fcstDate":"20260510","fcstTime":"0000","fcstValue":"0.1","nx":60,"ny":127},{"baseDate":"20260509","baseTime":"2300","category":"VVV","fcstDate":"20260510","fcstTime":"0000","fcstValue":"0","nx":


In [13]:
from datetime import datetime, timedelta

def get_weather_forecast(nx: int = 60, ny: int = 127) -> dict:
    """
    기상청 단기예보 조회
    nx, ny: 격자 좌표 (서울 중구 기본값)
    """
    url = "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst"
    
    # 기준 시간 계산 (가장 최근 발표 시각)
    now = datetime.now()
    
    # 단기예보 발표 시각: 02, 05, 08, 11, 14, 17, 20, 23시
    base_times = [2, 5, 8, 11, 14, 17, 20, 23]
    current_hour = now.hour
    
    # 가장 최근 발표 시각 찾기
    base_hour = max([t for t in base_times if t <= current_hour], default=23)
    
    if base_hour == 23 and current_hour < 23:
        base_date = (now - timedelta(days=1)).strftime("%Y%m%d")
        base_hour = 23
    else:
        base_date = now.strftime("%Y%m%d")
    
    base_time = f"{base_hour:02d}00"
    
    params = {
        "serviceKey": API_KEY,
        "numOfRows": 100,
        "pageNo": 1,
        "dataType": "JSON",
        "base_date": base_date,
        "base_time": base_time,
        "nx": nx,
        "ny": ny
    }
    
    print(f"📅 기준일시: {base_date} {base_time}")
    
    response = requests.get(url, params=params)
    data = response.json()
    
    return data


# 테스트 실행
weather_data = get_weather_forecast(60, 127)  # 서울 중구
print("\n📡 기상청 API 응답 (일부):")
print(json.dumps(weather_data, ensure_ascii=False, indent=2))

📅 기준일시: 20260510 0200

📡 기상청 API 응답 (일부):
{
  "response": {
    "header": {
      "resultCode": "00",
      "resultMsg": "NORMAL_SERVICE"
    },
    "body": {
      "dataType": "JSON",
      "items": {
        "item": [
          {
            "baseDate": "20260510",
            "baseTime": "0200",
            "category": "TMP",
            "fcstDate": "20260510",
            "fcstTime": "0300",
            "fcstValue": "12",
            "nx": 60,
            "ny": 127
          },
          {
            "baseDate": "20260510",
            "baseTime": "0200",
            "category": "UUU",
            "fcstDate": "20260510",
            "fcstTime": "0300",
            "fcstValue": "-0.3",
            "nx": 60,
            "ny": 127
          },
          {
            "baseDate": "20260510",
            "baseTime": "0200",
            "category": "VVV",
            "fcstDate": "20260510",
            "fcstTime": "0300",
            "fcstValue": "0.3",
            "nx": 60,
           

In [14]:
def parse_weather(data: dict) -> dict:
    """
    기상청 응답에서 현재 시각 기온, 체감온도 추출
    
    주요 카테고리:
    TMP  = 기온 (°C)
    WCI  = 체감온도 (°C)
    POP  = 강수확률 (%)
    WSD  = 풍속 (m/s)
    REH  = 습도 (%)
    """
    try:
        items = data["response"]["body"]["items"]["item"]
        
        # 카테고리별로 정리
        category_map = {}
        for item in items:
            cat = item["category"]
            if cat not in category_map:
                category_map[cat] = item["fcstValue"]
        
        result = {
            "기온(°C)": category_map.get("TMP", "-"),
            "체감온도(°C)": category_map.get("WCI", "-"),
            "강수확률(%)": category_map.get("POP", "-"),
            "풍속(m/s)": category_map.get("WSD", "-"),
            "습도(%)": category_map.get("REH", "-"),
            "하늘상태": category_map.get("SKY", "-"),  # 1:맑음 3:구름많음 4:흐림
        }
        return result
    
    except Exception as e:
        print(f"❌ 파싱 오류: {e}")
        return {}


# 파싱 결과 확인
parsed_weather = parse_weather(weather_data)
print("\n✅ 기상 파싱 결과:")
for key, value in parsed_weather.items():
    print(f"  {key}: {value}")


✅ 기상 파싱 결과:
  기온(°C): 12
  체감온도(°C): -
  강수확률(%): 0
  풍속(m/s): 0.6
  습도(%): 60
  하늘상태: 1
